In [1]:
!pip install reportlab pillow -q
print("✓ Listo")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.0 MB/s eta 0:00:00
✓ Listo


In [2]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, PageBreak, KeepTogether, Image
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY, TA_RIGHT
import os

print("✓ Importaciones correctas")

✓ Importaciones correctas


In [3]:
AZUL_OSCURO = colors.HexColor('#1D3557')
AZUL_MEDIO  = colors.HexColor('#457B9D')
ROJO        = colors.HexColor('#E63946')
VERDE       = colors.HexColor('#2D6A4F')
NARANJA     = colors.HexColor('#F4A261')
GRIS_CLARO  = colors.HexColor('#F8F9FA')
GRIS_BORDE  = colors.HexColor('#DEE2E6')
GRIS_TEXTO  = colors.HexColor('#6C757D')
AZUL_CLARO  = colors.HexColor('#EBF5FB')
AZUL_FILA   = colors.HexColor('#F1F8FF')

# ── Estilos de texto ──────────────────────────────────────
E_NORMAL = ParagraphStyle('Normal2',
    fontSize=10, fontName='Helvetica',
    textColor=colors.black,
    alignment=TA_JUSTIFY,
    spaceAfter=6, leading=14)

E_H1 = ParagraphStyle('H1',
    fontSize=16, fontName='Helvetica-Bold',
    textColor=AZUL_OSCURO,
    spaceBefore=18, spaceAfter=4, leading=20)

E_H2 = ParagraphStyle('H2',
    fontSize=12, fontName='Helvetica-Bold',
    textColor=AZUL_MEDIO,
    spaceBefore=12, spaceAfter=5, leading=15)

E_CAPTION = ParagraphStyle('Caption',
    fontSize=8, fontName='Helvetica-Oblique',
    textColor=GRIS_TEXTO,
    alignment=TA_CENTER,
    spaceAfter=10, leading=10)

E_NOTA = ParagraphStyle('Nota',
    fontSize=8, fontName='Helvetica',
    textColor=GRIS_TEXTO,
    alignment=TA_LEFT,
    spaceAfter=4, leading=11)

E_CELDA = ParagraphStyle('Celda',
    fontSize=9, fontName='Helvetica',
    textColor=colors.black,
    alignment=TA_LEFT,
    leading=12, wordWrap='CJK')

E_CELDA_C = ParagraphStyle('CeldaC',
    fontSize=9, fontName='Helvetica',
    textColor=colors.black,
    alignment=TA_CENTER,
    leading=12, wordWrap='CJK')

E_CELDA_HDR = ParagraphStyle('CeldaHdr',
    fontSize=9, fontName='Helvetica-Bold',
    textColor=colors.white,
    alignment=TA_CENTER,
    leading=12, wordWrap='CJK')

E_REF = ParagraphStyle('Ref',
    fontSize=9, fontName='Helvetica',
    textColor=colors.black,
    alignment=TA_JUSTIFY,
    leading=13, leftIndent=20,
    firstLineIndent=-20, spaceAfter=8)

E_PORTADA_T = ParagraphStyle('PortT',
    fontSize=26, fontName='Helvetica-Bold',
    textColor=colors.white,
    alignment=TA_CENTER,
    leading=32, spaceAfter=10)

E_PORTADA_S = ParagraphStyle('PortS',
    fontSize=13, fontName='Helvetica',
    textColor=AZUL_MEDIO,
    alignment=TA_CENTER,
    leading=18, spaceAfter=8)

E_RESUMEN = ParagraphStyle('Resumen',
    fontSize=9, fontName='Helvetica',
    textColor=colors.black,
    alignment=TA_JUSTIFY,
    leading=14)

E_LISTA = ParagraphStyle('Lista',
    fontSize=10, fontName='Helvetica',
    textColor=colors.black,
    alignment=TA_LEFT,
    leading=14, leftIndent=20,
    firstLineIndent=-12, spaceAfter=4)

print("✓ Estilos definidos")

✓ Estilos definidos


In [4]:
PAGE_W = 17*cm  # ancho útil con márgenes de 2cm

def seccion(texto):
    """Título H1 con línea decorativa."""
    return [
        Paragraph(texto, E_H1),
        HRFlowable(width='100%', thickness=2,
                   color=AZUL_MEDIO, spaceAfter=8),
    ]

def subseccion(texto):
    return [Paragraph(texto, E_H2)]

def p(texto):
    return [Paragraph(texto, E_NORMAL)]

def lista_numerada(items):
    out = []
    for i, item in enumerate(items, 1):
        out.append(Paragraph(f'<b>{i}.</b> {item}', E_LISTA))
    return out

def hallazgo(numero, texto):
    """Caja de hallazgo con wrap correcto."""
    data = [[
        Paragraph(f'<b>Hallazgo<br/>{numero}</b>',
                  ParagraphStyle('HN', fontSize=9,
                                 fontName='Helvetica-Bold',
                                 textColor=colors.white,
                                 alignment=TA_CENTER,
                                 leading=12)),
        Paragraph(texto,
                  ParagraphStyle('HT', fontSize=9,
                                 fontName='Helvetica',
                                 textColor=AZUL_OSCURO,
                                 alignment=TA_JUSTIFY,
                                 leading=13,
                                 wordWrap='CJK'))
    ]]
    t = Table(data, colWidths=[2.2*cm, 14.8*cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',  (0,0), (0,0), AZUL_MEDIO),
        ('BACKGROUND',  (1,0), (1,0), AZUL_CLARO),
        ('VALIGN',      (0,0), (-1,-1), 'MIDDLE'),
        ('PADDING',     (0,0), (-1,-1), 8),
        ('BOX',         (0,0), (-1,-1), 0.5, GRIS_BORDE),
        ('LINEAFTER',   (0,0), (0,-1), 0.5, GRIS_BORDE),
    ]))
    return [t, Spacer(1, 8)]

def tabla(encabezados, filas, anchos=None, caption=None,
          alinear_centro=None):
    """
    Tabla con wrap correcto en todas las celdas.
    alinear_centro: lista de índices de columna a centrar.
    """
    if anchos is None:
        anchos = [PAGE_W / len(encabezados)] * len(encabezados)
    if alinear_centro is None:
        alinear_centro = list(range(len(encabezados)))

    def celda(texto, es_header=False, centrar=True):
        estilo = E_CELDA_HDR if es_header else (
                 E_CELDA_C if centrar else E_CELDA)
        return Paragraph(str(texto), estilo)

    data = [[celda(h, es_header=True) for h in encabezados]]
    for i, fila in enumerate(filas):
        row = []
        for j, val in enumerate(fila):
            centrar = j in alinear_centro
            row.append(celda(val, centrar=centrar))
        data.append(row)

    t = Table(data, colWidths=anchos, repeatRows=1)
    n = len(filas)
    style = TableStyle([
        ('BACKGROUND',    (0,0), (-1,0),  AZUL_OSCURO),
        ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING',    (0,0), (-1,-1), 6),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
        ('LEFTPADDING',   (0,0), (-1,-1), 6),
        ('RIGHTPADDING',  (0,0), (-1,-1), 6),
        ('GRID',          (0,0), (-1,-1), 0.4, GRIS_BORDE),
    ])
    for i in range(1, n+1):
        bg = colors.white if i % 2 == 1 else AZUL_FILA
        style.add('BACKGROUND', (0,i), (-1,i), bg)
    t.setStyle(style)

    out = [t]
    if caption:
        out.append(Paragraph(caption, E_CAPTION))
    return out

def figura(ruta, caption, ancho=14*cm):
    """Imagen centrada con caption. Si no existe, muestra aviso."""
    out = []
    if os.path.exists(ruta):
        img = Image(ruta, width=ancho,
                    height=ancho*0.55,  # proporción 16:9 aprox
                    kind='proportional')
        img.hAlign = 'CENTER'
        out.append(Spacer(1, 6))
        out.append(img)
    else:
        out.append(Paragraph(
            f'[Figura no encontrada: {ruta}]', E_NOTA))
    out.append(Paragraph(caption, E_CAPTION))
    return out

def meta_row(clave, valor):
    return [
        Paragraph(f'<b>{clave}</b>',
                  ParagraphStyle('MK', fontSize=9,
                                 fontName='Helvetica-Bold',
                                 textColor=AZUL_OSCURO,
                                 leading=13)),
        Paragraph(valor,
                  ParagraphStyle('MV', fontSize=9,
                                 fontName='Helvetica',
                                 textColor=colors.black,
                                 leading=13))
    ]

print("✓ Funciones auxiliares definidas")

✓ Funciones auxiliares definidas


In [5]:
def portada():
    out = []

    # Bloque de título
    titulo_data = [[
        Paragraph('MERCADO LABORAL Y POBREZA<br/>EN ECUADOR 2007–2024',
                  E_PORTADA_T)
    ]]
    t_titulo = Table(titulo_data, colWidths=[PAGE_W])
    t_titulo.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), AZUL_OSCURO),
        ('TOPPADDING',    (0,0), (-1,-1), 35),
        ('BOTTOMPADDING', (0,0), (-1,-1), 35),
        ('LEFTPADDING',   (0,0), (-1,-1), 20),
        ('RIGHTPADDING',  (0,0), (-1,-1), 20),
    ]))
    out.append(t_titulo)
    out.append(Spacer(1, 0.6*cm))

    out.append(Paragraph(
        'Análisis integral: evolución, patrones y predicción '
        'mediante Machine Learning',
        E_PORTADA_S))
    out.append(Spacer(1, 0.4*cm))
    out.append(HRFlowable(width='100%', thickness=1,
                          color=GRIS_BORDE))
    out.append(Spacer(1, 0.5*cm))

    # Metadatos
    meta = [
        ('Autor',        'Bryan Anthony López Guerrero'),
        ('Formación',    'Ing. Tecnologías de la Información | Máster Visual Analytics '
                         'y Big Data | Esp. Big Data e IA'),
        ('Fuentes',      'INEC-ENEMDU | Banco Mundial | Banco Central del Ecuador'),
        ('Período',      '2007 – 2024'),
        ('Herramientas', 'Python · Power BI · Scikit-learn · Google Colab'),
        ('Fecha',        'Abril 2026'),
    ]
    meta_data = [meta_row(k, v) for k, v in meta]
    t_meta = Table(meta_data, colWidths=[3.8*cm, 13.2*cm])
    t_meta.setStyle(TableStyle([
        ('VALIGN',        (0,0), (-1,-1), 'TOP'),
        ('TOPPADDING',    (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ('LEFTPADDING',   (0,0), (-1,-1), 5),
        ('LINEBELOW',     (0,0), (-1,-1), 0.3, GRIS_BORDE),
        *[('BACKGROUND', (0,i), (-1,i),
           colors.white if i%2==0 else GRIS_CLARO)
          for i in range(len(meta))]
    ]))
    out.append(t_meta)
    out.append(Spacer(1, 0.8*cm))

    # Resumen ejecutivo
    resumen_data = [[
        Paragraph(
            '<b>Resumen ejecutivo</b><br/><br/>'
            'Este informe presenta un análisis integral del mercado laboral y la pobreza '
            'en Ecuador para el período 2007–2024, utilizando datos oficiales del '
            'INEC-ENEMDU, el Banco Mundial y el Banco Central del Ecuador. El estudio '
            'combina análisis exploratorio de datos, visualización interactiva en Power BI '
            'y un modelo de Machine Learning (Regresión Lineal, validación Leave-One-Out, '
            'MAE=0.72 pp, R²=0.903). Los hallazgos revelan que Ecuador redujo su pobreza '
            '15.2 pp entre 2007 y 2017, pero el COVID-19 revirtió 10.9 pp en un solo año. '
            'En 2024 la pobreza (28.0%) sigue 6.5 pp por encima del mínimo histórico. '
            'La informalidad estructural (58%), la brecha territorial (29.2 pp promedio) '
            'y el deterioro del ingreso laboral emergen como los factores más críticos.',
            E_RESUMEN)
    ]]
    t_res = Table(resumen_data, colWidths=[PAGE_W])
    t_res.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), AZUL_CLARO),
        ('TOPPADDING',    (0,0), (-1,-1), 14),
        ('BOTTOMPADDING', (0,0), (-1,-1), 14),
        ('LEFTPADDING',   (0,0), (-1,-1), 14),
        ('RIGHTPADDING',  (0,0), (-1,-1), 14),
        ('BOX',           (0,0), (-1,-1), 1, AZUL_MEDIO),
    ]))
    out.append(t_res)
    return out

print("✓ Portada definida")

✓ Portada definida


In [6]:
def seccion_1():
    out = []
    out += seccion('1. Introducción y Contexto')

    out += p('Ecuador enfrenta una crisis laboral estructural que se manifiesta en altos '
             'niveles de informalidad, una persistente brecha de pobreza entre zonas '
             'urbanas y rurales, y un deterioro sostenido de la calidad del empleo. '
             'Aunque las estadísticas oficiales muestran tasas de desempleo relativamente '
             'bajas, más del 58% de los trabajadores ecuatorianos operan en el sector '
             'informal, sin acceso a seguridad social ni ingresos estables.')
    out.append(Spacer(1, 4))

    out += p('La Encuesta Nacional de Empleo, Desempleo y Subempleo (ENEMDU) del INEC '
             'proporciona la fuente de datos más completa para el análisis del mercado '
             'laboral ecuatoriano. Complementada con indicadores del Banco Mundial y del '
             'Banco Central del Ecuador, esta base permite construir una visión integral '
             'de la relación entre condiciones laborales y niveles de pobreza.')

    out += subseccion('1.1 Justificación del estudio')
    out += p('El período 2007–2024 concentra eventos económicos de alto impacto: el boom '
             'petrolero (2007–2014), la caída del precio del petróleo (2015–2016), la '
             'pandemia de COVID-19 (2020–2021) y la recuperación frágil posterior '
             '(2022–2024). Analizar cómo cada uno de estos shocks afectó el mercado '
             'laboral y los niveles de pobreza permite identificar patrones estructurales '
             'que trascienden los ciclos económicos.')

    out += subseccion('1.2 Preguntas de investigación')
    out += lista_numerada([
        '¿Cómo ha evolucionado la tasa de pobreza por ingresos en Ecuador entre 2007 y 2024?',
        '¿Qué tan grande es la brecha de pobreza entre zonas urbanas y rurales?',
        '¿Qué relación existe entre la informalidad laboral y los niveles de pobreza?',
        '¿Cómo se relacionan el crecimiento del PIB y la reducción de la pobreza?',
        '¿Qué variables laborales y económicas predicen mejor la tasa de pobreza?',
    ])

    out += subseccion('1.3 Hipótesis de partida')
    out += lista_numerada([
        'H1: La informalidad laboral es el predictor más relevante de pobreza a nivel nacional.',
        'H2: La brecha urbano-rural en pobreza supera los 20 pp en todo el período.',
        'H3: El ingreso laboral promedio tiene correlación negativa fuerte con la pobreza.',
        'H4: Un modelo de regresión lineal predice la pobreza con error menor a 2 pp.',
    ])
    return out


def seccion_2():
    out = []
    out += seccion('2. Metodología')

    out += subseccion('2.1 Fuentes de datos')
    out += p('El estudio utiliza tres fuentes de datos oficiales complementarias:')
    out += tabla(
        ['Fuente', 'Indicadores principales', 'Período'],
        [
            ['INEC – ENEMDU',
             'Pobreza, informalidad, empleo, ingresos laborales, Gini',
             '2007–2024'],
            ['Banco Mundial',
             'PIB per cápita, crecimiento económico, empleo sectorial',
             '2000–2024'],
            ['Banco Central del Ecuador',
             'Salario básico unificado histórico',
             '2007–2024'],
        ],
        anchos=[4*cm, 9*cm, 4*cm],
        caption='Tabla 1. Fuentes de datos utilizadas en el estudio.',
        alinear_centro=[2])

    out += subseccion('2.2 Proceso de análisis')
    out += p('El análisis se realizó en cinco etapas secuenciales:')
    out += tabla(
        ['Etapa', 'Descripción', 'Herramienta'],
        [
            ['1. Obtención',
             'Descarga via API REST (Banco Mundial) y construcción manual '
             'desde boletines técnicos del INEC', 'Python – requests'],
            ['2. Limpieza',
             'Tratamiento de valores nulos, estandarización de variables '
             'y unión de múltiples fuentes', 'Python – pandas'],
            ['3. EDA',
             'Análisis exploratorio: tendencias, correlaciones, '
             'brechas territoriales y análisis estadístico',
             'matplotlib · seaborn'],
            ['4. Dashboard',
             'Visualización interactiva con filtros temporales, '
             'KPIs dinámicos y gráficos de proyección', 'Power BI'],
            ['5. Modelado',
             'Entrenamiento de 5 modelos predictivos con '
             'validación Leave-One-Out Cross Validation', 'scikit-learn'],
        ],
        anchos=[3*cm, 10*cm, 4*cm],
        caption='Tabla 2. Etapas del proceso de análisis.',
        alinear_centro=[0, 2])

    out += subseccion('2.3 Validación del modelo')
    out += p('Dado el tamaño reducido del dataset (N=14 años con datos completos), se '
             'utilizó Leave-One-Out Cross Validation (LOOCV) como técnica de validación. '
             'Este método entrena el modelo con N-1 observaciones y predice la restante '
             'en cada iteración, maximizando el uso de los datos disponibles. Es el '
             'estándar académico recomendado para series temporales cortas.')
    out += tabla(
        ['Modelo', 'MAE (pp)', 'RMSE (pp)', 'Seleccionado'],
        [
            ['Regresión Lineal', '0.72', '0.72', '✓ Sí'],
            ['Ridge (L2)',       '0.80', '0.80', 'No'],
            ['Lasso (L1)',       '1.26', '1.26', 'No'],
            ['Random Forest',    '1.50', '1.50', 'No'],
            ['Gradient Boosting','1.47', '1.47', 'No'],
        ],
        anchos=[5*cm, 4*cm, 4*cm, 4*cm],
        caption='Tabla 3. Comparación de modelos con validación LOOCV. '
                'Seleccionado: menor MAE.')
    return out

print("✓ Secciones 1 y 2 definidas")

✓ Secciones 1 y 2 definidas


In [7]:
def seccion_3():
    out = []
    out += seccion('3. Resultados y Hallazgos')

    # 3.1
    out += subseccion('3.1 Evolución de la pobreza 2007–2024')
    out += p('La tasa de pobreza por ingresos mostró una tendencia decreciente sostenida '
             'entre 2007 y 2017, pasando del 36.7% al 21.5% — una reducción de 15.2 '
             'puntos porcentuales en una década. Este período coincide con el auge de los '
             'ingresos petroleros que financiaron programas de inversión social. Sin '
             'embargo, el COVID-19 produce el mayor salto registrado: 10.9 pp en un solo '
             'año. En 2024 la pobreza se ubica en 28.0%, todavía 6.5 pp por encima '
             'del mínimo histórico de 2017.')
    out += figura('01_evolucion_pobreza.png',
                  'Figura 1. Evolución de la pobreza por ingresos en Ecuador 2007–2024. '
                  'Fuente: elaboración propia con datos INEC-ENEMDU.')
    out += tabla(
        ['Año', 'Pobreza total (%)', 'Pobreza extrema (%)', 'Variación anual (pp)'],
        [
            ['2007', '36.7', '16.5', '—'],
            ['2010', '32.8', '13.1', '-3.9'],
            ['2014', '22.5', '7.7',  '-2.6'],
            ['2017', '21.5', '8.4',  '-0.5'],
            ['2020', '32.4', '14.9', '+10.9'],
            ['2022', '25.2', '8.2',  '-3.6'],
            ['2024', '28.0', '12.7', '+2.8'],
        ],
        anchos=[3*cm, 4.5*cm, 4.5*cm, 5*cm],
        caption='Tabla 4. Evolución de la pobreza en años clave 2007–2024. '
                'Fuente: INEC-ENEMDU.')
    out += hallazgo(1,
        'Ecuador redujo su pobreza 15.2 pp entre 2007 y 2017, pero el COVID-19 '
        'revirtió 10.9 pp de esa ganancia en un solo año. En 2024 la pobreza (28.0%) '
        'sigue siendo 6.5 pp más alta que el mínimo histórico registrado en 2017 (21.5%).')

    # 3.2
    out += subseccion('3.2 Brecha territorial de la pobreza')
    out += p('Uno de los hallazgos más persistentes del análisis es la profunda desigualdad '
             'territorial en la distribución de la pobreza. En todos los años del período, '
             'la tasa de pobreza rural más que duplica a la urbana, con una brecha promedio '
             'de 29.2 puntos porcentuales. En 2024 esta brecha es de 22.4 pp (43.3% rural '
             'vs 20.9% urbana), lo que significa que un ecuatoriano del campo tiene más del '
             'doble de probabilidad de ser pobre que uno de la ciudad.')
    out += figura('02_brecha_urbano_rural.png',
                  'Figura 2. Brecha de pobreza urbano-rural en Ecuador 2007–2024. '
                  'Fuente: elaboración propia con datos INEC-ENEMDU.')
    out += tabla(
        ['Año', 'Pobreza urbana (%)', 'Pobreza rural (%)', 'Brecha (pp)'],
        [
            ['2007', '25.0', '61.5', '36.5'],
            ['2011', '17.4', '50.9', '33.5'],
            ['2014', '14.0', '43.0', '29.0'],
            ['2017', '15.0', '40.2', '25.2'],
            ['2020', '25.1', '49.9', '24.8'],
            ['2022', '17.8', '39.4', '21.6'],
            ['2024', '20.9', '43.3', '22.4'],
        ],
        anchos=[3*cm, 4.5*cm, 4.5*cm, 5*cm],
        caption='Tabla 5. Brecha de pobreza urbano-rural en años clave. '
                'Fuente: INEC-ENEMDU.')
    out += hallazgo(2,
        'La brecha rural-urbana promedió 29.2 pp en todo el período. Aunque se redujo '
        'de 36.5 pp en 2007 a 22.4 pp en 2024, sigue siendo estructuralmente alta '
        'y sin señales de cierre definitivo.')

    # 3.3
    out += subseccion('3.3 Informalidad y mercado laboral')
    out += p('La informalidad laboral es el rasgo más persistente del mercado laboral '
             'ecuatoriano. En 2024 alcanzó el 58.0%, el peor registro comparable desde '
             '2007, superando incluso el pico pandémico de 2020 (56.5%). El análisis '
             'de correlación muestra una relación positiva y estadísticamente significativa '
             'entre informalidad y pobreza (R²=0.43, p=0.011): cada punto porcentual '
             'adicional de informalidad se asocia con 1.03 pp más de pobreza.')
    out += figura('03_informalidad_vs_pobreza.png',
                  'Figura 3. Relación entre informalidad laboral y pobreza en Ecuador '
                  '2011–2024. Fuente: elaboración propia con datos INEC-ENEMDU.')
    out += tabla(
        ['Año', 'Informalidad (%)', 'Empleo adecuado (%)',
         'Desempleo (%)', 'Ingreso laboral (USD)'],
        [
            ['2011', '58.8', '41.2', '4.2', '318'],
            ['2014', '53.0', '47.0', '3.8', '379'],
            ['2017', '53.5', '42.3', '4.6', '375'],
            ['2019', '53.0', '38.8', '3.8', '374'],
            ['2020', '56.5', '31.4', '5.0', '321'],
            ['2022', '54.3', '36.8', '3.8', '368'],
            ['2024', '58.0', '34.1', '3.4', '355'],
        ],
        anchos=[2.5*cm, 3.5*cm, 4*cm, 3*cm, 4*cm],
        caption='Tabla 6. Indicadores del mercado laboral en años clave. '
                'Fuente: INEC-ENEMDU.')
    out += hallazgo(3,
        'La informalidad laboral alcanzó el 58.0% en 2024. Cada punto porcentual '
        'adicional de informalidad se asocia con 1.03 pp más de pobreza '
        '(R²=0.43, p=0.011, estadísticamente significativo).')

    # 3.4
    out += subseccion('3.4 Relación PIB y pobreza')
    out += p('La correlación entre PIB per cápita y tasa de pobreza es fuerte (R²=0.73), '
             'confirmando que el crecimiento económico está asociado con la reducción de '
             'la pobreza. Sin embargo, la relación es asimétrica: las caídas del PIB '
             'producen aumentos de pobreza más rápidos de lo que el crecimiento los reduce. '
             'El año 2020 es el caso extremo: una caída del PIB de 9.2% generó un aumento '
             'de pobreza de 10.9 pp.')
    out += figura('04_pib_vs_pobreza.png',
                  'Figura 4. Relación entre crecimiento económico y pobreza en Ecuador. '
                  'Fuente: elaboración propia con datos INEC-ENEMDU y Banco Mundial.')
    out += hallazgo(4,
        'La relación PIB-pobreza es asimétrica: las recesiones aumentan la pobreza más '
        'rápidamente de lo que el crecimiento la reduce. El PIB per cápita explica el '
        '73% de la variación histórica de la pobreza (R²=0.73).')
    return out


def seccion_4():
    out = []
    out += seccion('4. Modelo Predictivo')

    out += subseccion('4.1 Selección y desempeño del modelo')
    out += p('El modelo de Regresión Lineal fue seleccionado entre los cinco modelos '
             'evaluados por tener el menor error de predicción (MAE=0.72 pp). Con un '
             'R²=0.903 calculado sobre las predicciones LOOCV concatenadas, el modelo '
             'explica el 90.3% de la variación histórica de la pobreza usando únicamente '
             'seis variables laborales y económicas. El error promedio de 0.72 pp significa '
             'que el modelo predice, por ejemplo, una pobreza de entre 27.3% y 28.7% '
             'cuando el valor real es 28.0% — una precisión muy alta para el contexto '
             'de series temporales cortas.')

    out += subseccion('4.2 Importancia de variables')
    out += p('El análisis de importancia de variables revela que el ingreso laboral '
             'promedio es el predictor más poderoso de la tasa de pobreza (importancia: '
             '0.481), casi 2.4 veces más importante que la segunda variable. Esto tiene '
             'una interpretación económica clara: en Ecuador la pobreza no es '
             'principalmente un problema de falta de empleo (desempleo 3.4% en 2024), '
             'sino de calidad e ingreso del trabajo disponible.')
    out += figura('08_importancia_variables.png',
                  'Figura 5. Importancia relativa de variables predictoras de pobreza. '
                  'Modelo: Regresión Lineal. Fuente: elaboración propia.',
                  ancho=13*cm)
    out += tabla(
        ['Variable', 'Importancia', 'Categoría', 'Interpretación'],
        [
            ['Ingreso laboral promedio', '0.481', 'Ingresos',
             'Variable más crítica: determina si los ingresos superan la línea de pobreza'],
            ['Salario básico unificado', '0.204', 'Política salarial',
             'El piso salarial impacta directamente en los ingresos del sector formal'],
            ['Coeficiente Gini', '0.164', 'Desigualdad',
             'Mayor desigualdad se asocia con mayor pobreza estructural'],
            ['Tasa de desempleo', '0.069', 'Mercado laboral',
             'Impacto moderado, consistente con baja tasa de desempleo estructural'],
            ['PIB per cápita', '0.049', 'Economía',
             'Captura el efecto macroeconómico general sobre los ingresos'],
            ['Tasa de informalidad', '0.035', 'Mercado laboral',
             'Alta colinealidad con ingreso laboral reduce su importancia relativa'],
        ],
        anchos=[3.8*cm, 2.2*cm, 3*cm, 8*cm],
        caption='Tabla 7. Importancia relativa de variables predictoras. '
                'Fuente: elaboración propia.',
        alinear_centro=[0, 1, 2])
    out += hallazgo(5,
        'El ingreso laboral promedio explica el 48.1% de la capacidad predictiva del '
        'modelo. La pobreza en Ecuador es principalmente un problema de calidad e '
        'ingreso del empleo, no de cantidad de empleo disponible.')

    out += subseccion('4.3 Proyecciones 2025')
    out += p('Utilizando el modelo entrenado con datos históricos 2011–2024, se proyectaron '
             'tres escenarios para la tasa de pobreza en 2025, basados en supuestos sobre '
             'la evolución de las seis variables predictoras:')
    out += figura('09_proyeccion_pobreza.png',
                  'Figura 6. Proyección de pobreza en Ecuador 2025 bajo tres escenarios. '
                  'Fuente: elaboración propia.')
    out += tabla(
        ['Escenario', 'Pobreza proyectada',
         'Ingreso laboral supuesto', 'Gini supuesto', 'Informalidad supuesta'],
        [
            ['Optimista', '25.9%', '$370 USD/mes', '0.465', '56.0%'],
            ['Base',      '28.3%', '$355 USD/mes', '0.478', '57.5%'],
            ['Pesimista', '30.6%', '$340 USD/mes', '0.492', '60.0%'],
        ],
        anchos=[3*cm, 3.5*cm, 4*cm, 3*cm, 3.5*cm],
        caption='Tabla 8. Proyecciones de pobreza Ecuador 2025 bajo tres escenarios.')
    out += hallazgo(6,
        'El escenario base proyecta una pobreza de 28.3% para 2025, prácticamente igual '
        'al nivel de 2024 (28.0%). Sin mejoras estructurales en ingresos laborales y '
        'reducción de desigualdad, Ecuador no logrará reducir su pobreza en el corto plazo.')
    return out

print("✓ Secciones 3 y 4 definidas")

✓ Secciones 3 y 4 definidas


In [8]:
def seccion_5():
    out = []
    out += seccion('5. Conclusiones y Recomendaciones')

    out += subseccion('5.1 Conclusiones')
    conclusiones = [
        ('<b>La pobreza en Ecuador es estructural, no coyuntural.</b> '
         'Aunque el país logró reducciones importantes entre 2007 y 2017, la '
         'recuperación post-COVID es significativamente más lenta que el deterioro '
         'durante la crisis. Esto sugiere que los factores que impulsaron la reducción '
         'inicial (ingresos petroleros, inversión social masiva) no son sostenibles '
         'y que las causas estructurales permanecen sin resolver.'),
        ('<b>La informalidad laboral es el problema central del mercado laboral.</b> '
         'Con más del 58% en informalidad en 2024 y una correlación directa con la '
         'pobreza (R²=0.43, p=0.011), la calidad del empleo — no su cantidad — es la '
         'variable crítica a intervenir. Las políticas de creación de empleo sin '
         'mejoras en formalización y remuneración son insuficientes.'),
        ('<b>La brecha territorial persiste sin señales de cierre estructural.</b> '
         'La desigualdad urbano-rural promedió 29.2 pp durante el período analizado. '
         'Las políticas de reducción de pobreza han tenido mayor efectividad en zonas '
         'urbanas, dejando a las comunidades rurales en situación de vulnerabilidad '
         'persistente que requiere intervenciones diferenciadas por territorio.'),
        ('<b>El crecimiento económico es necesario pero no suficiente.</b> '
         'La correlación PIB-pobreza (R²=0.73) confirma que el crecimiento ayuda, '
         'pero la distribución importa más. El Gini se mantuvo por encima de 0.45 '
         'durante todo el período, indicando que los beneficios del crecimiento no '
         'se distribuyen equitativamente entre la población.'),
    ]
    for i, c in enumerate(conclusiones, 1):
        out.append(Paragraph(f'<b>{i}.</b> {c}',
                             ParagraphStyle('Conc', fontSize=10,
                                           fontName='Helvetica',
                                           alignment=TA_JUSTIFY,
                                           leading=14, spaceAfter=8,
                                           leftIndent=12)))

    out += subseccion('5.2 Recomendaciones de política pública')
    out += p('Basándose en los hallazgos del análisis, se proponen las siguientes '
             'recomendaciones para reducir la pobreza en Ecuador de forma sostenida:')
    out += tabla(
        ['#', 'Recomendación', 'Variable objetivo'],
        [
            ['1',
             'Implementar incentivos tributarios para la formalización laboral, '
             'especialmente en sectores de alta informalidad (agricultura, comercio)',
             'Informalidad laboral'],
            ['2',
             'Fortalecer programas de capacitación laboral orientados a incrementar '
             'la productividad y el ingreso en zonas rurales',
             'Ingreso laboral rural'],
            ['3',
             'Revisar la política de salario básico con criterios de productividad '
             'sectorial y costo de vida regional',
             'Salario básico'],
            ['4',
             'Expandir la cobertura de transferencias monetarias condicionadas '
             'en zonas rurales con pobreza superior al 40%',
             'Pobreza rural'],
            ['5',
             'Crear un sistema de monitoreo trimestral de indicadores de calidad '
             'del empleo a nivel provincial',
             'Empleo adecuado / Gini'],
        ],
        anchos=[0.8*cm, 12*cm, 4.2*cm],
        caption='Tabla 9. Recomendaciones de política pública basadas en los hallazgos.',
        alinear_centro=[0, 2])
    return out


def seccion_6():
    out = []
    out += seccion('6. Referencias')
    refs = [
        'Instituto Nacional de Estadística y Censos – INEC. (2025). <i>Encuesta Nacional '
        'de Empleo, Desempleo y Subempleo – ENEMDU. Boletines técnicos 2007–2024.</i> '
        'Quito: INEC. Recuperado de https://www.ecuadorencifras.gob.ec',

        'Instituto Nacional de Estadística y Censos – INEC. (2025). <i>Guía de uso '
        'de base de datos ENEMDU 2021–2025.</i> Quito: INEC.',

        'Instituto Nacional de Estadística y Censos – INEC. (2025). <i>Boletín Técnico '
        'N° 02-2025-ENEMDU: Pobreza y desigualdad, diciembre 2024.</i> Quito: INEC.',

        'Banco Mundial. (2024). <i>Ecuador – Indicadores de desarrollo mundial.</i> '
        'Washington D.C.: Banco Mundial. https://datos.bancomundial.org',

        'Banco Mundial. (2025). <i>Pobreza y empleo en Ecuador: dos caras de una misma '
        'realidad.</i> Blog de la Práctica Global de Pobreza y Equidad, LAC.',

        'Banco Central del Ecuador – BCE. (2024). <i>Series históricas: salario básico '
        'unificado 2007–2024.</i> Quito: BCE. https://estadisticas.bce.fin.ec',

        'Cobena Rodríguez, P. J., & Palacios Cedeño, N. (2024). El desempleo y el '
        'índice de pobreza en el Ecuador. <i>Ciencia Latina Revista Científica '
        'Multidisciplinar, 8</i>(3). https://doi.org/10.37811/cl_rcm.v8i3.11994',

        'Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. '
        '<i>Journal of Machine Learning Research, 12</i>, 2825–2830.',
    ]
    for i, r in enumerate(refs, 1):
        out.append(Paragraph(f'[{i}] {r}', E_REF))
    return out

print("✓ Secciones 5 y 6 definidas")

✓ Secciones 5 y 6 definidas


In [9]:
def generar_pdf(nombre='informe_final_ecuador_empleo_pobreza.pdf'):

    doc = SimpleDocTemplate(
        nombre,
        pagesize=A4,
        rightMargin=2*cm,
        leftMargin=2*cm,
        topMargin=2.5*cm,
        bottomMargin=2.5*cm,
        title='Mercado Laboral y Pobreza en Ecuador 2007-2024',
        author='Bryan Anthony López Guerrero',
    )

    contenido = []

    # Portada
    contenido += portada()
    contenido.append(PageBreak())

    # Secciones
    contenido += seccion_1()
    contenido.append(PageBreak())

    contenido += seccion_2()
    contenido.append(PageBreak())

    contenido += seccion_3()
    contenido.append(PageBreak())

    contenido += seccion_4()
    contenido.append(PageBreak())

    contenido += seccion_5()
    contenido.append(PageBreak())

    contenido += seccion_6()

    doc.build(contenido)
    print(f"✓ PDF generado: {nombre}")
    return nombre

nombre = generar_pdf()

from google.colab import files
files.download(nombre)
print("✓ Descargado — revisa tu carpeta de descargas")

✓ PDF generado: informe_final_ecuador_empleo_pobreza.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Descargado — revisa tu carpeta de descargas
